#Internship Project

In [ ]:
import pandas as pd

file_path = '/content/expanded_annotated_labeled_reddit_posts.xlsx'
df = pd.read_excel(file_path)
display(df.head())

,id,title,selftext,created_utc,author,url,score,num_comments,combined_text,category,sub_category,help_seeking
0,1lk106c,Where do you buy your custom food wrapping pap...,"Hey everyone,\n\nI‚Äôm curious ‚Äî if you run ...",2025-06-25 09:21:32,West_Necessary1831,https://www.reddit.com/r/foodtrucks/comments/1...,3.0,2,Where do you buy your custom food wrapping pap...,Equipment,Equipment Sourcing/Purchasing,YES
1,1ljys8f,Charcuterie Cart - What permits needed?,Researching a little into food carts and stuff...,2025-06-25 06:52:05,First_Pass_3375,https://i.redd.it/pa6i66oct09f1.jpeg,0.0,3,Charcuterie Cart - What permits needed?\n\nRes...,Business Operations,Licensing & Permits,YES
2,1ljr6k2,Not everything is lighting!,"So I'm running 2 fryers 90kbtus each, a 36"" gr...",2025-06-25 00:03:13,idkfoodtrucks,https://www.reddit.com/r/foodtrucks/comments/1...,2.0,6,Not everything is lighting!\n\nSo I'm running ...,Equipment,Gas/Plumbing Setup,YES
3,1ljqv6k,Freezer,"Hello, we are located in Canada and we are in ...",2025-06-24 23:48:37,Aromatic-Finance6428,https://www.reddit.com/r/foodtrucks/comments/1...,3.0,5,"Freezer\n\nHello, we are located in Canada and...",Equipment,Cold Storage,YES
4,1ljpzhv,Central Florida Space Available to Rent,I don‚Äôt know if this is allowed in the group...,2025-06-24 23:10:10,asdf_lkjh123,https://www.reddit.com/r/foodtrucks/comments/1...,0.0,0,Central Florida Space Available to Rent\n\nI d...,Business Operations,Rent,NO


Automatically Extract Keywords per Subcategory

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
keyword_rows = []

for subcat in df["sub_category"].unique():
    texts = df[df["sub_category"] == subcat]["combined_text"].dropna()

    if len(texts) < 3:
        continue  # avoid weak subcategories

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=10
    )

    tfidf = vectorizer.fit_transform(texts)
    keywords = vectorizer.get_feature_names_out()

    for word in keywords:
        keyword_rows.append({
            "sub_category": subcat,
            "keyword": word
        })

keywords_df = pd.DataFrame(keyword_rows)
display(keywords_df)

,sub_category,keyword
0,Equipment Sourcing/Purchasing,exhaust
1,Equipment Sourcing/Purchasing,fan
2,Equipment Sourcing/Purchasing,food
3,Equipment Sourcing/Purchasing,hood
4,Equipment Sourcing/Purchasing,hoodmart
...,...,...
305,Suppliers,ice
306,Suppliers,sell
307,Suppliers,serve
308,Suppliers,suppliers


Assign Arbitrary numbers

In [ ]:
#
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

rows = []

for subcat in df["sub_category"].unique():
    texts = df[df["sub_category"] == subcat]["combined_text"].dropna()

    if len(texts) < 3:
        continue

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=10
    )

    tfidf_matrix = vectorizer.fit_transform(texts)
    scores = tfidf_matrix.mean(axis=0).A1
    keywords = vectorizer.get_feature_names_out()

    # Scale scores to 1–100
    scaled_scores = np.interp(scores, (scores.min(), scores.max()), (1, 100))

    for kw, score in zip(keywords, scaled_scores):
        rows.append({
            "sub_category": subcat,
            "keyword": kw,
            "arbitrary_value": int(round(score))
        })

keyword_table = pd.DataFrame(rows)
display(keyword_table)



,sub_category,keyword,arbitrary_value
0,Equipment Sourcing/Purchasing,exhaust,56
1,Equipment Sourcing/Purchasing,fan,100
2,Equipment Sourcing/Purchasing,food,78
3,Equipment Sourcing/Purchasing,hood,15
4,Equipment Sourcing/Purchasing,hoodmart,1
...,...,...,...
305,Suppliers,ice,33
306,Suppliers,sell,100
307,Suppliers,serve,87
308,Suppliers,suppliers,33


Clasify post to two groups

In [ ]:
#Categorize the two groups first
STARTUP_SUBCATEGORIES = {
    "Licensing & Permits",
    "Business Planning",
    "Startup Costs",
    "Financing",
    "Equipment Sourcing"
}

OPERATIONAL_SUBCATEGORIES = {
    "Cold Storage",
    "Gas/Plumbing Setup",
    "Equipment Maintenance",
    "Staffing",
    "Marketing",
    "Supply Chain",
    "Operations & Logistics",
    "Financial Performance"
}

import re

def classify_post(
    post_text: str,
    keyword_table: list,
    min_score_threshold: int = 30
):
    """
    Classifies a Reddit post into:
    - Pre-Startup Help
    - Operational Help
    - Unclear / Mixed

    Parameters:
    - post_text (str): Title + body of the post
    - keyword_table (list of dicts):
        [{sub_category, keyword, arbitrary_value}]
    - min_score_threshold (int): minimum score to count as signal

    Returns:
    - dict with classification results
    """

    text = post_text.lower()

    startup_score = 0
    operational_score = 0
    matched_keywords = []

    for row in keyword_table:
        keyword = row["keyword"].lower()
        score = row["arbitrary_value"]
        subcat = row["sub_category"]

        if re.search(rf"\b{keyword}\b", text):
            matched_keywords.append((keyword, subcat, score))

            if subcat in STARTUP_SUBCATEGORIES:
                startup_score += score
            elif subcat in OPERATIONAL_SUBCATEGORIES:
                operational_score += score

    # Apply threshold
    if startup_score < min_score_threshold:
        startup_score = 0
    if operational_score < min_score_threshold:
        operational_score = 0

    # Final decision
    if startup_score > operational_score:
        label = "Pre-Startup Help"
    elif operational_score > startup_score:
        label = "Operational Help"
    else:
        label = "Unclear / Mixed"

    return {
        "label": label,
        "startup_score": startup_score,
        "operational_score": operational_score,
        "matched_keywords": matched_keywords
    }



In [ ]:
sample_post_text = "I need help with my business plan and getting permits for my new food truck."
classification_result = classify_post(sample_post_text, keyword_table.to_dict('records'))
display(classification_result)

{'label': 'Pre-Startup Help',
 'startup_score': 480,
 'operational_score': 0,
 'matched_keywords': [('food', 'Equipment Sourcing/Purchasing', 78),
  ('business', 'Licensing & Permits', 36),
  ('food', 'Licensing & Permits', 100),
  ('permits', 'Licensing & Permits', 22),
  ('food', 'Rent', 1),
  ('food', 'Branding', 71),
  ('truck', 'Branding', 100),
  ('food', 'General Event Planning', 46),
  ('truck', 'General Event Planning', 34),
  ('food', 'Profit Margins / Cost Control', 100),
  ('truck', 'Profit Margins / Cost Control', 7),
  ('food', 'Start-Up', 100),
  ('need', 'Start-Up', 7),
  ('truck', 'Start-Up', 76),
  ('food', 'Hiring', 100),
  ('need', 'Hiring', 2),
  ('truck', 'Hiring', 48),
  ('food', 'Advice for New Operators', 100),
  ('truck', 'Advice for New Operators', 48),
  ('truck', 'Generator/Power Solutions', 43),
  ('food', 'Daily Operation', 100),
  ('truck', 'Daily Operation', 56),
  ('food', 'Uncategorized', 100),
  ('truck', 'Uncategorized', 45),
  ('business', 'Vehicle

In [ ]:
import re
keyword_table = [# ---- Startup: Licensing & Permits ----
    {"sub_category": "Licensing & Permits", "keyword": "permit", "value": 100},
    {"sub_category": "Licensing & Permits", "keyword": "license", "value": 90},
    {"sub_category": "Licensing & Permits", "keyword": "inspection", "value": 80},
    {"sub_category": "Licensing & Permits", "keyword": "health department", "value": 85},

    # ---- Startup: Business Planning ----
    {"sub_category": "Business Planning", "keyword": "start", "value": 70},
    {"sub_category": "Business Planning", "keyword": "business plan", "value": 90},
    {"sub_category": "Business Planning", "keyword": "new food truck", "value": 85},

    # ---- Operational: Equipment Issues ----
    {"sub_category": "Equipment Maintenance", "keyword": "broken", "value": 90},
    {"sub_category": "Equipment Maintenance", "keyword": "repair", "value": 85},
    {"sub_category": "Equipment Maintenance", "keyword": "malfunction", "value": 80},

    # ---- Operational: Cold Storage ----
    {"sub_category": "Cold Storage", "keyword": "freezer", "value": 100},
    {"sub_category": "Cold Storage", "keyword": "refrigerator", "value": 90},
    {"sub_category": "Cold Storage", "keyword": "refrigeration", "value": 85},
]




# Subcategory Groups

STARTUP_SUBCATEGORIES = {
    "Licensing & Permits",
    "Business Planning",
    "Startup Costs",
    "Financing",
    "Equipment Sourcing"
}

OPERATIONAL_SUBCATEGORIES = {
    "Cold Storage",
    "Equipment Maintenance",
    "Staffing",
    "Marketing",
    "Supply Chain",
    "Operations & Logistics",
    "Financial Performance"
}

# Classification Function
def classify_post(text, keyword_table, threshold=30):
    text = str(text).lower()

    startup_score = 0
    operational_score = 0

    for row in keyword_table:
        keyword = row["keyword"].lower()
        value = row["value"]
        subcat = row["sub_category"]

        if re.search(rf"\b{re.escape(keyword)}\b", text):
            if subcat in STARTUP_SUBCATEGORIES:
                startup_score += value
            elif subcat in OPERATIONAL_SUBCATEGORIES:
                operational_score += value

    # Apply minimum threshold
    if startup_score < threshold:
        startup_score = 0
    if operational_score < threshold:
        operational_score = 0

    # Final decision
    if startup_score > operational_score:
        return "Pre-Startup Help"
    elif operational_score > startup_score:
        return "Operational Help"
    else:
        return "Unclear / Mixed"


# Batch Classification

df["intent_group"] = df["combined_text"].apply(
    lambda x: classify_post(x, keyword_table)
)
output_path = "classified_food_truck_posts.xlsx"
df.to_excel(output_path, index=False)

print("Batch classification completed.")
print(df["intent_group"].value_counts())


Batch classification completed.
intent_group
Unclear / Mixed     155
Pre-Startup Help     36
Operational Help     10
Name: count, dtype: int64
